In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pip install -q pydantic==2.12.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 13.0 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [3]:
!pip install -q rwkv==0.8.31

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.9/410.9 kB 8.7 MB/s eta 0:00:00ta 0:00:01


In [4]:
!pip install -qi https://test.pypi.org/simple/ rwkvx==0.2.0.dev8

In [5]:
!pip install -q rwkv huggingface_hub

In [6]:
import os

import huggingface_hub

In [7]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

In [8]:
from rwkvx.generation import (
    RWKVSession,
    RWKVTokenizer,
    RWKVModel,
    RWKVSampler,
    GenerationConfig
)

from rwkvx.conversation import (
    History,
    ConversationManager,
    Message,
    RWKV7PromptRenderer,
    get_default_rwkv7_prompt_templates,
)

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv_cuda...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv_cuda/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/4] c++ -MMD -MF gemm_fp16_cublas.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/gemm_fp16_cublas.cpp -o gemm_fp16_cublas.o 
[2/4] c++ -MMD -MF wrapper.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-p

Loading extension module wkv_cuda...
Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv7s...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv7s/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv7s...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/3] c++ -MMD -MF rwkv7_op.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/rwkv7_op.cpp -o rwkv7_op.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output rwkv7.cuda.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include

Loading extension module wkv7s...


In [9]:
# model_title = "rwkv7-g1b-2.9b-20251205-ctx8192"
# model_title = "rwkv7-g0b-7.2b-20251220-ctx8192"
model_title = "rwkv7-g0b-13.3b-20251130-ctx8192"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")

rwkv7-g0b-13.3b-20251130-ctx8192.pth:   0%|          | 0.00/26.5G [00:00<?, ?B/s]

In [10]:
config = GenerationConfig(
    max_new_tokens=1000,
    stop_tokens=[0, 261],  # 0, \n\n
)

print(config)

GenerationConfig(max_new_tokens=1000, temperature=1.0, top_p=0.3, top_k=0, presence_penalty=0.5, frequency_penalty=0.5, decay_penalty=0.996, stop_tokens=[0, 261])


In [11]:
model = RWKVModel(model_path=model_path)

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/fac6e8b05751a73f51422536be922156d1b5b06c/rwkv7-g0b-13.3b-20251130-ctx8192 (cuda fp16)



OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 100.19 MiB is free. Process 4097 has 14.64 GiB memory in use. Of the allocated memory 14.51 GiB is allocated by PyTorch, and 20.14 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
tokenizer = RWKVTokenizer(model)
sampler = RWKVSampler(config)

In [ ]:
templates = get_default_rwkv7_prompt_templates()
renderer = RWKV7PromptRenderer(templates)

mgr = ConversationManager(renderer)

In [ ]:
session = RWKVSession(model, tokenizer, sampler, mgr, config)

In [ ]:
session.reset()

In [ ]:
mgr.add_system('Include emojis to all your answers.')
mgr.build_prompt()

In [ ]:
mgr.add_user('Tell me a short joke.')
mgr.build_prompt()

In [ ]:
joke2 = session.generate_message()
print(joke2)

In [ ]:
mgr.build_prompt()

In [ ]:
mgr.add_user('Tell me a joke different from previous one.')
mgr.build_prompt()

In [ ]:
joke = session.generate_message()
print(joke)

In [ ]:
mgr.build_prompt()

In [ ]:
mgr.add_user('Tell me more complex joke.')
mgr.build_prompt()

In [ ]:
joke3 = session.generate_message()
print(joke3)

In [ ]:
mgr.build_prompt()

In [ ]:
mgr.add_user('List topics of all jokes that you have told me. Explain the second joke.', metadata={'think_mode': ' think'})
mgr.build_prompt()

In [ ]:
final_answer = session.generate_message()

In [ ]:
print(final_answer.metadata['think_text'])

In [ ]:
print(final_answer.text)

In [ ]:
mgr.build_prompt()

In [ ]:
mgr.add_user('Lets wrap up. Thank you!')
mgr.build_prompt()

In [ ]:
session.generate_message()

In [ ]:
mgr.add_user('Wait. It seems that you have missed the joke about mathematicians')
session.generate_message()